# Notebook 7 - Run 4 LLM Enrichment

*  Adds DynamoDB merchant lookup (Pass 1 + Pass 2) to Run 3 pipeline.

In [ ]:
#
# New in Run 4 vs Run 3:
#   - Step 1.5: DB lookup on merchant_name field (Pass 1, free)
#   - Step 6a:  LLM NER to extract merchant candidate from description
#   - Step 6b:  DB lookup on LLM-extracted merchant name (Pass 2, free)
#   - Step 6c:  LLM classification with all hints (replaces Step 6)
#   - Output:   20 fields including p1_*/p2_* merchant columns
#   - Config:   RUN_NER_ALWAYS flag for comparison vs production mode
#   - Fix:      AWS_BEARER_TOKEN_BEDROCK popped after load_dotenv
#
# Sections:
#   1. Setup and imports
#   2. Config
#   3. Data load and df_clean build
#   4. Context loading
#   5. Pre-processing pipeline
#   6. DynamoDB client
#   7. Prompt builders
#   8. Bedrock call functions
#   9. Stratified sample
#  10. Batch loop
#  11. Post-process and evaluation
# =============================================================================

In [ ]:
# -----------------------------------------------------------------------------
# 1. Setup and imports
# -----------------------------------------------------------------------------
import os, re, json, time
import numpy as np
import pandas as pd
import boto3
from pathlib import Path
from typing import Optional
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(dotenv_path=r'/home/sagemaker-user/swtest1/llm_poc/.env', override=True)

# Remove stale Bedrock token injected by .env - must happen before any boto3 call
for _k in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN', 'AWS_BEARER_TOKEN_BEDROCK']:
    os.environ.pop(_k, None)

DEV_ROLE = os.getenv('DEV_ROLE')
print(f'Imports OK | DEV_ROLE loaded: {bool(DEV_ROLE)}')

In [ ]:
# -----------------------------------------------------------------------------
# 2. Config
# -----------------------------------------------------------------------------
from run4_prompt_config import (
    MODEL_ID, MAX_TOKENS, NER_TOKENS, TEMPERATURE, BUDGET_USD,
    RUN_NER_ALWAYS,
    TARGET_ROWS, BASE_LAYER_ROWS, TOP_UP_ROWS, TOP_UP_PER_CAT, FOCUS_CATEGORIES,
    AGGREGATOR_DT_VALUE, MCC_SKIP, MCC_NON_SPEND, BILLER_CODE_ZEROS,
    PROVIDER_ANZ, PROVIDER_ING, PROVIDER_BANKWEST, PROVIDER_UP, PROVIDER_MACQUARIE,
    INVALID_KEY_BLOCKLIST, RULES_FILE, RULES_STRIP_PATTERN,
    OUTPUT_DIR, OUTPUT_FILE,
)
print(f'Config loaded | RUN_NER_ALWAYS={RUN_NER_ALWAYS} | Output -> {OUTPUT_FILE}')

In [ ]:
# -----------------------------------------------------------------------------
# 3. Data load and df_clean build  (unchanged from Run 3)
# -----------------------------------------------------------------------------
df_sample = pd.read_parquet('data/sample_data3.parquet')
print(f'df_sample: {len(df_sample):,} rows')

df_sample['x3_processed'] = df_sample['properties'].str.contains(
    'ml_transfer_sub_category_key', case=False, na=False
)
df_work = df_sample[~df_sample['x3_processed']].copy()

mask_remove = (
    (df_work['base_category_key'] == 'SAVINGS') |
    (df_work['base_category_key'] == 'UNCATEGORISED') |
    (
        (df_work['base_category_key'] == 'INTERNAL_TRANSFER') &
        df_work['original_description'].str.contains(r'(?i)round.?up', na=False)
    )
)
df_clean = df_work[~mask_remove].copy()
print(f'df_clean: {len(df_clean):,} rows')

In [ ]:
# -----------------------------------------------------------------------------
# 4. Context loading  (unchanged from Run 3)
# -----------------------------------------------------------------------------
cat_keys_df  = pd.read_csv('assets/cat_keys.csv')
VALID_KEYS   = set(cat_keys_df['base_category_key'].dropna().str.strip())

biller_df  = pd.read_csv('assets/biller_mapping.csv')
biller_map = dict(zip(
    biller_df.iloc[:, 0].astype(str).str.strip(),
    biller_df.iloc[:, 1].astype(str).str.strip()
))

mcc_df  = pd.read_csv('assets/mcc_mapping.csv')
mcc_map = dict(zip(
    mcc_df.iloc[:, 0].astype(str).str.strip(),
    mcc_df.iloc[:, 1].astype(str).str.strip()
))

ns_rules = pd.concat([
    pd.read_csv('assets/layered_ns_rules.csv'),
    pd.read_csv('assets/new_bank_patterns.csv'),
], ignore_index=True)

tax_df = pd.read_csv('assets/taxonomy_master_updated_override.csv')
tax_df['spend_type'] = tax_df.apply(
    lambda r: r['manual_spend_override']
    if pd.notna(r.get('manual_spend_override', '')) and r['manual_spend_override'] != ''
    else r['spend_type'], axis=1
)

def build_taxonomy_context(df):
    lines = ['VALID BASE_CATEGORY_KEY VALUES\n']
    for stype in ['spend', 'non_spend']:
        subset = df[df['spend_type'] == stype]
        lines.append(f'--- {stype.upper()} ---')
        gcol = 'category_group' if stype == 'spend' else 'category_type'
        if gcol in df.columns:
            for grp, gdf in subset.groupby(gcol, dropna=False):
                keys = ', '.join(sorted(gdf['base_category_key'].dropna().unique()))
                lines.append(f'  {grp}: {keys}')
        lines.append('')
    return '\n'.join(lines)

TAXONOMY_CONTEXT = build_taxonomy_context(tax_df)

def build_gt_spend_map(df):
    result = {}
    for _, row in df.iterrows():
        key = str(row.get('base_category_key', '')).strip()
        st  = str(row.get('spend_type', '')).strip()
        if st == 'mixed':
            ov = str(row.get('manual_spend_override', '')).strip()
            ln = str(row.get('leans', '')).strip()
            st = ov if ov and ov != 'nan' else ln
        result[key] = st if st and st != 'nan' else None
    return result

GT_SPEND_MAP = build_gt_spend_map(tax_df)

rules_raw     = Path(RULES_FILE).read_text()
sec3          = re.search(RULES_STRIP_PATTERN, rules_raw, re.IGNORECASE | re.MULTILINE)
RULES_CONTEXT = rules_raw[:sec3.start()].strip() if sec3 else rules_raw

print(f'Valid keys: {len(VALID_KEYS)} | Taxonomy: {len(TAXONOMY_CONTEXT):,} chars')


In [ ]:
# -----------------------------------------------------------------------------
# 5. Pre-processing pipeline  (unchanged from Run 3)
# -----------------------------------------------------------------------------
def _str(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return ''
    return str(val).strip()

def resolve_biller(row):
    bc = _str(row.get('biller_code', ''))
    if bc in BILLER_CODE_ZEROS: return None
    return biller_map.get(bc)

def resolve_mcc(row):
    mcc = _str(row.get('merchant_category_code', ''))
    if mcc in MCC_SKIP: return None, False
    hit = mcc_map.get(mcc)
    return hit, (hit is not None)

def run_decision_tree(row):
    agg = _str(row.get('aggregator_id', row.get('aggregator', '')))
    if agg != AGGREGATOR_DT_VALUE: return 'none', None
    tt, mcc, bc, bn, mn = (_str(row.get(f, '')) for f in
        ['transaction_type', 'merchant_category_code', 'biller_code', 'biller_name', 'merchant_name'])
    prov = _str(row.get('provider_name', '')).upper()
    try: amount = float(row.get('amount', 0) or 0)
    except: amount = 0.0
    bc_zero = bc in BILLER_CODE_ZEROS
    mcc_ok  = mcc and mcc not in MCC_SKIP
    if tt in {'0','1','2'}:                         return 'non_spend', 'Rule 1'
    if bc and not bc_zero:                          return 'spend',     'Rule 2'
    if bn:                                          return 'spend',     'Rule 3'
    if mcc in MCC_NON_SPEND:                        return 'non_spend', 'Rule 4'
    if mcc and amount == 0.0:                       return 'non_spend', 'Rule 5'
    if mcc_ok and PROVIDER_MACQUARIE not in prov:   return 'spend',     'Rule 6'
    if PROVIDER_ANZ in prov and mn:                 return 'spend',     'Rule 9'
    if PROVIDER_ANZ in prov and bc_zero:            return 'non_spend', 'Rule 10'
    if PROVIDER_ING in prov and mn:                 return 'spend',     'Rule 11'
    if PROVIDER_BANKWEST in prov and mn:            return 'spend',     'Rule 12'
    if PROVIDER_UP in prov and mn:                  return 'spend',     'Rule 13'
    return 'ambiguous', None

_NS_TOKENS = re.compile(
    r'\b(ATO|CENTRELINK|MEDICARE|MYOB|BPAY|DIRECT DEBIT|PAYROLL|SALARY|'
    r'REFUND|REBATE|INTEREST|DIVIDEND|TRANSFER|PAYMENT FROM|PAYMENT TO|'
    r'DIRECT CREDIT|GOVERNMENT|PENSION|BENEFITS?|SUPERANNUATION)\b', re.IGNORECASE
)

def detect_merchant_presence(row, mcc_proxy=False):
    mn   = _str(row.get('merchant_name', ''))
    desc = _str(row.get('original_description', ''))
    if mn and not _NS_TOKENS.search(mn): return True
    if mcc_proxy: return True
    if desc and not _NS_TOKENS.search(desc):
        if 1 <= len(desc.split()) <= 6: return True
    return False

def run_ns_regex(row):
    desc = _str(row.get('original_description', '')).upper()
    prov = _str(row.get('provider_name', '')).upper()
    for _, rule in ns_rules.iterrows():
        pattern  = _str(rule.get('pattern', ''))
        cat      = _str(rule.get('base_category_key', rule.get('category_type', ''))).upper()
        provider = _str(rule.get('provider', ''))
        if not pattern or not cat: continue
        if provider and provider.upper() not in prov: continue
        try:
            if re.search(pattern, desc, re.IGNORECASE): return cat
        except re.error: continue
    return None

def run_pipeline(row):
    biller_key = resolve_biller(row)
    if biller_key:
        return {
            'biller_hit': True, 'biller_category_key': biller_key,
            'mcc_category_key': None, 'spend_type_hint': 'spend',
            'dt_rule': None, 'merchant_present': True,
            'regex_category_type': None, 'execution_path_steps': ['biller'],
        }
    mcc_key, mcc_proxy   = resolve_mcc(row)
    spend_hint, dt_rule  = run_decision_tree(row)
    merchant_present     = detect_merchant_presence(row, mcc_proxy=mcc_proxy)
    is_ns = (spend_hint == 'non_spend') or (not merchant_present and spend_hint in {'none', 'ambiguous'})
    regex_cat = run_ns_regex(row) if is_ns else None
    steps = []
    if mcc_key:              steps.append('mcc')
    if spend_hint != 'none': steps.append('dt')
    if merchant_present:     steps.append('merchant')
    if regex_cat:            steps.append('regex')
    return {
        'biller_hit': False, 'biller_category_key': None,
        'mcc_category_key': mcc_key, 'spend_type_hint': spend_hint,
        'dt_rule': dt_rule, 'merchant_present': merchant_present,
        'regex_category_type': regex_cat, 'execution_path_steps': steps,
    }

print('Pre-processing pipeline ready')

In [ ]:
# -----------------------------------------------------------------------------
# 6. DynamoDB client
# Refresh STS credentials once at batch start. Token valid ~1hr.
# Call refresh_dynamodb_client() if running >45 min batches.
# -----------------------------------------------------------------------------
def refresh_dynamodb_client():
    sts     = boto3.client('sts')
    session = sts.assume_role(RoleArn=DEV_ROLE, RoleSessionName='run4-session')
    creds   = session['Credentials']
    return boto3.client(
        'dynamodb', region_name='ap-southeast-2',
        aws_access_key_id=creds['AccessKeyId'],
        aws_secret_access_key=creds['SecretAccessKey'],
        aws_session_token=creds['SessionToken'],
    )

dynamodb_client = refresh_dynamodb_client()
print('DynamoDB client ready')

def lookup_merchant(merchant_name: str) -> dict:
    """Lookup normalised merchant string in DynamoDB. Returns hit dict or miss dict."""
    if not merchant_name: return {'found': False}
    normalised = merchant_name.upper().strip()
    escaped    = normalised.translate(str.maketrans({"'": "''"}))
    try:
        resp  = dynamodb_client.execute_statement(
            Statement=(
                "SELECT * FROM mapping "
                "WHERE \"type\" = 'merchant' "
                f"AND \"entity\" = '{escaped}'"
            )
        )
        items = resp.get('Items', [])
        if items:
            def unwrap(v): return next(iter(v.values())) if isinstance(v, dict) and v else None
            item = items[0]
            return {
                'found':    True,
                'merchant': unwrap(item.get('mapping')),   # canonical name
                'category': unwrap(item.get('category')),  # base_category_key
            }
        return {'found': False}
    except Exception as e:
        return {'found': False, 'error': str(e)}


In [ ]:
# -----------------------------------------------------------------------------
# 7. Prompt builders
# -----------------------------------------------------------------------------
def build_system_prompt():
    return f"""You are a financial transaction categorisation engine.
Your task is to assign a single base_category_key from the taxonomy below.

## OUTPUT FORMAT
Return a single JSON object with exactly these fields:
{{
  "spend_type": "spend | non_spend",
  "category_group": "string (spend only, else null)",
  "category_type": "string (non_spend only, else null)",
  "base_category_key": "EXACT_KEY_FROM_TAXONOMY",
  "base_category_key_naive": "EXACT_KEY_FROM_TAXONOMY",
  "confidence": 7,
  "reason": "keyword:value->keyword:value->KEY",
  "execution_path": "step->step->llm",
  "flag": null
}}
Return ONLY the JSON object. No explanation, no text before or after.

## CONFIDENCE SCALE (integer 1-10)
9-10: biller or DB merchant lookup hit
7-8:  MCC hit or strong DT signal
5-6:  merchant present + reasonable inference
3-4:  LLM intuition only, no upstream hints

## REASON FORMAT
Keyword chain only. Examples:
- "mcc:5411->merchant:Woolworths->GROCERIES"
- "dt:non_spend->regex:TRANSFERS->INTERNAL_TRANSFER"
- "no_hints->description:netflix->SUBSCRIPTIONS_RENEWALS"

## HINT PRIORITY ORDER
1. p1_db_category or p2_db_category [critical] - follow unless description clearly conflicts
2. mcc_category_key [high]
3. spend_type_hint from DT [high]
4. merchant_present [medium]
5. regex_category_type [medium]
6. Your own merchant/brand knowledge [low-medium]

## INVALID KEYS - NEVER RETURN THESE
{', '.join(INVALID_KEY_BLOCKLIST)}

## RULES
{RULES_CONTEXT}

## TAXONOMY
{TAXONOMY_CONTEXT}

## FLAG
If mcc_category_key and p2_db_category are both present and disagree, 
set flag type to "signal_conflict" with detail showing both signals 
e.g. "mcc:5969->GENERAL_MERCHANDISE vs db:Netflix->SUBSCRIPTIONS_RENEWALS".
Otherwise set flag to null.

"""

SYSTEM_PROMPT = build_system_prompt()
print(f'System prompt: {len(SYSTEM_PROMPT):,} chars')

NER_SYSTEM_PROMPT = """Extract the merchant or brand name from a bank transaction description.
Return ONLY a JSON object: {"merchant": "Normalised Name"}
- Title case, no location suffixes, no legal entity suffixes
- e.g. "Woolworths" not "WOOLWORTHS METRO SYDNEY PTY LTD"
- Return {"merchant": null} if no merchant identifiable (transfers, fees, government payments)
No explanation, no text before or after."""

def build_ner_prompt(row):
    return f"description: {_str(row.get('original_description', ''))}"

def build_user_prompt(row, signals, p1_db_category=None, p2_db_category=None):
    def _hint(val, label, tier):
        return f'{label}: {val} [{tier}]' if val else f'{label}: none'
    p2_tier = 'high' if signals.get('mcc_category_key') else 'critical'
    lines = [
        f"provider:               {_str(row.get('provider_name', ''))}",
        f"description:            {_str(row.get('original_description', ''))}",
        f"merchant_name:          {_str(row.get('merchant_name', ''))}",
        f"amount:                 {row.get('amount', '')}",
        f"transaction_type:       {_str(row.get('transaction_type', ''))}",
        f"account_type:           {_str(row.get('account_type', ''))}",
        f"merchant_category_code: {_str(row.get('merchant_category_code', ''))}",
        f"biller_code:            {_str(row.get('biller_code', ''))}",
        '',
        'Pipeline signals:',
        f"  {_hint(signals.get('biller_category_key'), 'biller_category_key', 'critical')}",
        f"  {_hint(p1_db_category, 'p1_db_category', 'critical')}",
        f"  {_hint(p2_db_category, 'p2_db_category', p2_tier)}",
        f"  {_hint(signals.get('mcc_category_key'), 'mcc_category_key', 'high')}",
        f"  spend_type_hint: {signals.get('spend_type_hint', 'none')} [high]"
            + (f" ({signals['dt_rule']})" if signals.get('dt_rule') else ''),
        f"  merchant_present: {signals.get('merchant_present', False)} [medium]",
        f"  {_hint(signals.get('regex_category_type'), 'regex_category_type', 'medium')}",
    ]
    return '\n'.join(lines)

NAIVE_SYSTEM_PROMPT = f"""You are a financial transaction categorisation engine.
Given only a transaction description and amount, assign the most likely base_category_key.
Return ONLY: {{"base_category_key_naive": "KEY"}}

## INVALID KEYS - NEVER RETURN THESE
{', '.join(INVALID_KEY_BLOCKLIST)}

## VALID KEYS
{TAXONOMY_CONTEXT}
"""

def build_naive_prompt(row):
    return f"description: {_str(row.get('original_description', ''))}\namount: {row.get('amount', '')}"

print('Prompt builders ready')

In [ ]:
# -----------------------------------------------------------------------------
# 8. Bedrock call functions
# Fresh client per call - required for SageMaker RefreshableCredentials
# -----------------------------------------------------------------------------
def call_bedrock(system_prompt, user_prompt, max_tokens=MAX_TOKENS, max_retries=4):
    client = boto3.client('bedrock-runtime', region_name='ap-southeast-2')
    body   = json.dumps({
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': max_tokens,
        'temperature': TEMPERATURE,
        'system': system_prompt,
        'messages': [{'role': 'user', 'content': user_prompt}],
    })
    for attempt in range(max_retries):
        try:
            resp = client.invoke_model(
                modelId=MODEL_ID, contentType='application/json',
                accept='application/json', body=body,
            )
            return json.loads(resp['body'].read())['content'][0]['text']
        except Exception as e:
            if 'ThrottlingException' in str(type(e)) and attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise

def parse_json(text):
    cleaned = re.sub(r'^```(?:json)?\s*', '', text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r'\s*```$', '', cleaned.strip())
    m = re.search(r'\{.*\}', cleaned, re.DOTALL)
    return json.loads(m.group() if m else cleaned)

print('Bedrock functions ready')

In [ ]:
# -----------------------------------------------------------------------------
# 9. Stratified sample  (unchanged from Run 3)
# -----------------------------------------------------------------------------
providers     = df_clean['provider_name'].dropna().unique()
rows_per_prov = max(1, BASE_LAYER_ROWS // len(providers))
base_frames   = [
    df_clean[df_clean['provider_name'] == p].sample(n=min(rows_per_prov, len(df_clean[df_clean['provider_name'] == p])), random_state=42)
    for p in providers
]
df_base = pd.concat(base_frames).drop_duplicates()

topup_frames = []
for cat in FOCUS_CATEGORIES:
    cat_df = df_clean[(df_clean['base_category_key'] == cat) & (~df_clean.index.isin(df_base.index))]
    n = min(TOP_UP_PER_CAT, len(cat_df))
    if n > 0: topup_frames.append(cat_df.sample(n=n, random_state=42))

df_topup      = pd.concat(topup_frames).drop_duplicates() if topup_frames else pd.DataFrame()
df_sample_run = pd.concat([df_base, df_topup]).drop_duplicates().reset_index(drop=True)
print(f'Sample: {len(df_sample_run)} rows across {len(providers)} providers')

In [ ]:
# -----------------------------------------------------------------------------
# 10. Batch loop
# -----------------------------------------------------------------------------
def process_row(idx, row):
    result = {
        # Passthrough
        'row_idx':               idx,
        'original_description':  row.get('original_description', ''),
        'merchant_name':         row.get('merchant_name', ''),
        'gt_base_category_key':  row.get('base_category_key', ''),
        'gt_spend_type': 'spend' if int(row.get('prop_ideas_has_merch', -1)) == 1 else 'non_spend' if int(row.get('prop_ideas_has_merch', -1)) == 0 else '',
        'gt_category_type':      row.get('prop_ideas_categorisation_name_nonspend', ''),
        # Step 1.5 - DB lookup on merchant_name
        'p1_db_merchant':        None,
        'p1_db_category':        None,
        # Step 6a - LLM NER
        'p2_llm_merchant':       'skipped',
        # Step 6b - DB lookup on LLM merchant
        'p2_db_merchant':        'skipped',
        'p2_db_category':        'skipped',
        # Step 6c - LLM classification
        'spend_type':            None,
        'category_group':        None,
        'category_type':         None,
        'base_category_key':     None,
        'base_category_key_naive': None,
        'confidence':            None,
        'reason':                None,
        'execution_path':        None,
        'flag':                  None,
        # Eval
        'parse_error':           False,
        'invalid_key':           False,
    }

    signals = run_pipeline(row)

    # Step 1 - biller short-circuit
    if signals['biller_hit']:
        key = signals['biller_category_key']
        result.update({
            'base_category_key':        key,
            'base_category_key_naive':  key,
            'spend_type':               'spend',
            'execution_path':           'biller->llm',
            'confidence':               10,
            'reason':                   f'biller:{key}',
        })
        return result

    # Step 1.5 - DB lookup on merchant_name field
    mn = _str(row.get('merchant_name', ''))
    if mn:
        p1 = lookup_merchant(mn)
        if p1['found']:
            result['p1_db_merchant'] = p1.get('merchant') or ''
            result['p1_db_category'] = p1.get('category') or ''
        else:
            result['p1_db_merchant'] = 'na'
            result['p1_db_category'] = 'na'

    p1_db_category = result['p1_db_category']

    # Step 6a - LLM NER (always if RUN_NER_ALWAYS, else only when both Pass 1 and MCC missed)
    mcc_hit = signals.get('mcc_category_key') is not None
    p1_hit = result['p1_db_category'] not in (None, 'na')

    #run_ner = (
    #    (RUN_NER_ALWAYS and p1_hit and not mcc_hit) or
    #    (not p1_hit and not mcc_hit)
    #)
    run_ner = True

    if run_ner:
        try:
            ner_raw    = call_bedrock(NER_SYSTEM_PROMPT, build_ner_prompt(row), max_tokens=NER_TOKENS)
            ner_parsed = parse_json(ner_raw)
            result['p2_llm_merchant'] = ner_parsed.get('merchant')
        except Exception:
            result['p2_llm_merchant'] = None

    # Step 6b - DB lookup on LLM merchant
    p2_db_category = None
    if result['p2_llm_merchant']:
        p2 = lookup_merchant(result['p2_llm_merchant'])
        if p2['found']:
            result['p2_db_merchant'] = p2.get('merchant') or ''
            result['p2_db_category'] = p2.get('category') or ''
        else:
            result['p2_db_merchant'] = 'na'
            result['p2_db_category'] = 'na'

    # Step 6c - LLM classification (parallel with naive)
    user_prompt  = build_user_prompt(row, signals, p1_db_category, p2_db_category)
    naive_prompt = build_naive_prompt(row)

    with ThreadPoolExecutor(max_workers=2) as ex:
        fut_full  = ex.submit(call_bedrock, SYSTEM_PROMPT,       user_prompt,  MAX_TOKENS)
        fut_naive = ex.submit(call_bedrock, NAIVE_SYSTEM_PROMPT, naive_prompt, 100)
        raw_full  = fut_full.result()
        raw_naive = fut_naive.result()

    # Parse naive
    try:
        result['base_category_key_naive'] = parse_json(raw_naive).get('base_category_key_naive')
    except Exception:
        pass

    # Parse full
    try:
        parsed = parse_json(raw_full)
        result.update({
            'spend_type':           parsed.get('spend_type'),
            'category_group':       parsed.get('category_group'),
            'category_type':        parsed.get('category_type'),
            'base_category_key':    parsed.get('base_category_key'),
            'confidence':           parsed.get('confidence'),
            'reason':               parsed.get('reason'),
            'execution_path':       parsed.get('execution_path') or
                                    '->'.join(signals['execution_path_steps']) + '->llm',
            'flag':                 json.dumps(parsed['flag']) if parsed.get('flag') else None,
        })
    except Exception as e:
        result['parse_error'] = True
        result['reason']      = f'PARSE_ERROR: {e}'
        return result

    #print(f"DEBUG conflict check: mcc={signals.get('mcc_category_key')} p2_db={result.get('p2_db_category')}")
    
    # Signal conflict check - MCC vs p2 DB
    if (result.get('p2_db_category') not in (None, 'na', 'skipped') and
        signals.get('mcc_category_key') and
        result['p2_db_category'] != signals['mcc_category_key']):
        result['flag'] = json.dumps({
            'type': 'signal_conflict',
            'detail': f"mcc:{signals['mcc_category_key']} vs p2_db:{result['p2_db_category']}"
        })

    pred = result.get('base_category_key', '')
    if pred in INVALID_KEY_BLOCKLIST or pred not in VALID_KEYS:
        result['invalid_key'] = True

    return result


# Smoke test
smoke = process_row(0, df_sample_run.iloc[0].to_dict())
print('Smoke test:')
for k, v in smoke.items():
    print(f'  {k}: {v}')

In [ ]:
results, errors = [], 0
processed_idx   = set()
start_time      = time.time()
JSONL_FILE      = OUTPUT_FILE.with_suffix('.jsonl')
CSV_SAVE_EVERY  = 50  # save CSV every N rows

if OUTPUT_FILE.exists():
    prev          = pd.read_csv(OUTPUT_FILE)
    processed_idx = set(prev['row_idx'].tolist())
    results       = prev.to_dict('records')
    print(f'Resuming - {len(processed_idx)} rows already done')

for i, (_, row) in enumerate(df_sample_run.iterrows()):
    if i in processed_idx: continue
    try:
        res = process_row(i, row.to_dict())
    except Exception as e:
        print(f'[{i}] EXCEPTION: {e}')
        errors += 1
        res = {'row_idx': i, 'parse_error': True, 'reason': str(e)}

    results.append(res)

    # JSONL - append single row, no rewrite
    with open(JSONL_FILE, 'a') as f:
        f.write(json.dumps(res) + '\n')

    # CSV - periodic save only
    if (i + 1) % CSV_SAVE_EVERY == 0 or i == len(df_sample_run) - 1:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

    if (i + 1) % 25 == 0 or i == len(df_sample_run) - 1:
        elapsed   = time.time() - start_time
        remaining = (elapsed / max(i + 1 - len(processed_idx), 1)) * (len(df_sample_run) - i - 1)
        print(f'[{i+1}/{len(df_sample_run)}] errors={errors} elapsed={elapsed:.0f}s est_remaining={remaining:.0f}s')

df_results = pd.DataFrame(results)
print(f'\nBatch complete: {len(df_results)} rows, {errors} errors | Output: {OUTPUT_FILE}')

In [ ]:
# -----------------------------------------------------------------------------
# 11. Post-process and evaluation
# -----------------------------------------------------------------------------
if 'df_results' not in dir():
    df_results = pd.read_csv(OUTPUT_FILE)

df_eval = df_results.copy()
#df_eval['gt_spend_type'] = df_eval['gt_base_category_key'].map(GT_SPEND_MAP)

# L1 - direct GT label
df_l1  = df_eval[df_eval['gt_spend_type'].isin(['spend', 'non_spend']) & df_eval['spend_type'].notna()]
l1_acc = (df_l1['spend_type'] == df_l1['gt_spend_type']).mean()

# L2a - direct GT label (non-spend only)
df_l2a  = df_eval[df_eval['gt_category_type'].notna() & (df_eval['gt_category_type'] != '') & df_eval['category_type'].notna()]
l2a_acc = (df_l2a['category_type'] == df_l2a['gt_category_type']).mean() if len(df_l2a) else 0

# L3
df_l3  = df_eval.dropna(subset=['gt_base_category_key', 'base_category_key'])
l3_acc = (df_l3['base_category_key'] == df_l3['gt_base_category_key']).mean()

# Naive vs pipeline lift
df_lift      = df_eval.dropna(subset=['gt_base_category_key', 'base_category_key', 'base_category_key_naive'])
pipeline_l3  = (df_lift['base_category_key']      == df_lift['gt_base_category_key']).mean()
naive_l3     = (df_lift['base_category_key_naive'] == df_lift['gt_base_category_key']).mean()

# Pass 1 and Pass 2 accuracy
df_p1   = df_eval[df_eval['p1_db_category'].notna()]
p1_acc  = (df_p1['p1_db_category'] == df_p1['gt_base_category_key']).mean() if len(df_p1) else 0
df_p2   = df_eval[df_eval['p2_db_category'].notna()]
p2_acc  = (df_p2['p2_db_category'] == df_p2['gt_base_category_key']).mean() if len(df_p2) else 0

print('=' * 50)
print('RUN 4 RESULTS SUMMARY')
print('=' * 50)
print(f'Total rows:            {len(df_results)}')
print(f'Parse errors:          {df_results["parse_error"].sum()} ({df_results["parse_error"].mean()*100:.1f}%)')
print(f'Invalid keys:          {df_results["invalid_key"].sum()} ({df_results["invalid_key"].mean()*100:.1f}%)')
print(f'L1 spend_type:         {l1_acc*100:.1f}% ({len(df_l1)} evaluable)')
print(f'L2a category_type:     {l2a_acc*100:.1f}% ({len(df_l2a)} evaluable)')
print(f'L3 exact key:          {l3_acc*100:.1f}% ({len(df_l3)} evaluable)')
print(f'Naive L3:              {naive_l3*100:.1f}%')
print(f'Pipeline lift:         +{(pipeline_l3-naive_l3)*100:.1f}pp')
print(f'Pass 1 coverage:       {len(df_p1)} rows ({len(df_p1)/len(df_results)*100:.1f}%)')
print(f'Pass 1 accuracy:       {p1_acc*100:.1f}%')
print(f'Pass 2 coverage:       {len(df_p2)} rows ({len(df_p2)/len(df_results)*100:.1f}%)')
print(f'Pass 2 accuracy:       {p2_acc*100:.1f}%')
print(f'Output:                {OUTPUT_FILE}')